# 05 — Nuclide Inventories
Mass evolution of key actinides and fission products in driver and blanket zones.
Tracks U-235/238 depletion, Pu vector buildup, minor actinide accumulation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

inv = pd.read_parquet('../results/inventories.parquet')
print('Materials:', inv['material'].unique())
print('Nuclides: ', inv['nuclide'].unique())
inv.head()

In [ ]:
# Pu vector evolution in inner driver fuel — one permutation
tag = inv['run_tag'].unique()[0]
driver = inv[(inv['run_tag'] == tag) & (inv['material'].str.contains('inner'))]

pu_nucs = ['Pu238', 'Pu239', 'Pu240', 'Pu241', 'Pu242']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for nuc in pu_nucs:
    sub = driver[driver['nuclide'] == nuc]
    if not sub.empty:
        axes[0].plot(sub['time_days'], sub['mass_g'] / 1e3, label=nuc, linewidth=1.5)
axes[0].set_xlabel('Time (days)')
axes[0].set_ylabel('Mass (kg)')
axes[0].set_title(f'Pu vector — inner driver ({tag})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# U-235 / U-238 depletion
for nuc, ls in [('U235', '--'), ('U238', '-')]:
    sub = driver[driver['nuclide'] == nuc]
    if not sub.empty:
        axes[1].plot(sub['time_days'], sub['mass_g'] / 1e3, label=nuc, linestyle=ls, linewidth=1.5)
axes[1].set_xlabel('Time (days)')
axes[1].set_ylabel('Mass (kg)')
axes[1].set_title(f'Uranium depletion — inner driver ({tag})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/05_inventories.png', dpi=150)
plt.show()

In [ ]:
# Cross-permutation comparison: total Pu-239 in blankets at end of cycle
final_time = inv['time_days'].max()
pu239_final = (
    inv[(inv['nuclide'] == 'Pu239')
        & (inv['material'].str.contains('blanket'))
        & (inv['time_days'] == final_time)]
    .groupby(['run_tag', 'material'])['mass_g']
    .sum()
    .reset_index()
)
print('Pu-239 in blankets at end of cycle (g):')
print(pu239_final.to_string(index=False))